# 20 — JUnit 5 Testing

## Objectives
- Write unit tests with JUnit 5
- Use lifecycle annotations: `@BeforeEach`, `@AfterEach`
- Write parameterized tests
- Apply AAA pattern (Arrange-Act-Assert)
- Test exceptions and edge cases

## JUnit 5 Annotations
| Annotation | Purpose |
|-----------|----------|
| `@Test` | Mark as test |
| `@DisplayName` | Human-readable test name |
| `@BeforeEach` | Run before each test |
| `@AfterEach` | Run after each test |
| `@BeforeAll` | Run once before all tests |
| `@ParameterizedTest` | Run with multiple values |
| `@ValueSource` | Inline test data |

In [1]:
// Example: Testing a BankAccount class
class BankAccount {
    private final String id;
    private double balance;
    private boolean frozen;
    
    BankAccount(String id, double balance) {
        if (balance < 0) throw new IllegalArgumentException("Balance cannot be negative");
        this.id = id; this.balance = balance;
    }
    
    boolean deposit(double amount) {
        if (frozen || amount <= 0) return false;
        balance += amount; return true;
    }
    
    boolean withdraw(double amount) {
        if (frozen || amount <= 0 || amount > balance) return false;
        balance -= amount; return true;
    }
    
    void freeze() { frozen = true; }
    double getBalance() { return balance; }
    String getId() { return id; }
}

// Simulating JUnit tests inline
BankAccount acc = new BankAccount("TEST-001", 1000.0);

// Test deposit
assert acc.deposit(500.0) : "Deposit should succeed";
assert acc.getBalance() == 1500.0 : "Balance should be 1500";

// Test withdrawal
assert acc.withdraw(200.0) : "Withdraw should succeed";
assert acc.getBalance() == 1300.0 : "Balance should be 1300";

// Test overdraft
assert !acc.withdraw(5000.0) : "Overdraft should fail";
assert acc.getBalance() == 1300.0 : "Balance unchanged after failed withdrawal";

// Test frozen account
acc.freeze();
assert !acc.deposit(100.0) : "Frozen account deposit should fail";
assert !acc.withdraw(100.0) : "Frozen account withdrawal should fail";

// Test invalid creation
try {
    new BankAccount("BAD", -100.0);
    System.out.println("FAIL: Should have thrown exception");
} catch (IllegalArgumentException e) {
    System.out.println("PASS: Negative balance rejected");
}

System.out.println("All inline assertions passed!");
System.out.println("Final balance: INR " + acc.getBalance());
System.out.println("\nIn real JUnit 5:");
System.out.println("  @Test void testDeposit() {");
System.out.println("      // Arrange");
System.out.println("      BankAccount acc = new BankAccount('T001', 1000.0);");
System.out.println("      // Act");
System.out.println("      boolean result = acc.deposit(500.0);");
System.out.println("      // Assert");
System.out.println("      assertTrue(result);");
System.out.println("      assertEquals(1500.0, acc.getBalance(), 0.01);");
System.out.println("  }");

PASS: Negative balance rejected
All inline assertions passed!
Final balance: INR 1000.0

In real JUnit 5:
  @Test void testDeposit() {
      // Arrange
      BankAccount acc = new BankAccount('T001', 1000.0);
      // Act
      boolean result = acc.deposit(500.0);
      // Assert
      assertTrue(result);
      assertEquals(1500.0, acc.getBalance(), 0.01);
  }


## Mini Challenge
Write JUnit 5 tests for the `StringUtils` class covering: `isPalindrome`, `reverse`, `isAnagram`, and edge cases (null, empty string).

In [5]:
import java.util.Arrays;

// 1. The Target Class (StringUtils)
class StringUtils {
    
    public static boolean isPalindrome(String str) {
        if (str == null) return false;
        String cleaned = str.replaceAll("\\s+", "").toLowerCase();
        String reversed = reverse(cleaned);
        return cleaned.equals(reversed);
    }
    
    public static String reverse(String str) {
        if (str == null) return null;
        return new StringBuilder(str).reverse().toString();
    }
    
    public static boolean isAnagram(String str1, String str2) {
        if (str1 == null || str2 == null) return false;
        
        String s1 = str1.replaceAll("\\s+", "").toLowerCase();
        String s2 = str2.replaceAll("\\s+", "").toLowerCase();
        
        if (s1.length() != s2.length()) return false;
        
        char[] array1 = s1.toCharArray();
        char[] array2 = s2.toCharArray();
        Arrays.sort(array1);
        Arrays.sort(array2);
        
        return Arrays.equals(array1, array2);
    }
}

// 2. Built-in Test Suite (No External Imports Required)
class StringUtilsTestRunner {
    private static int passed = 0;
    private static int failed = 0;

    public static void runTests() {
        System.out.println("=== RUNNING STRINGUTILS TEST SUITE ===");
        
        // --- isPalindrome Tests ---
        verify(StringUtils.isPalindrome("racecar") == true, "isPalindrome: racecar");
        verify(StringUtils.isPalindrome("Radar") == true, "isPalindrome: Radar");
        verify(StringUtils.isPalindrome("A man a plan a canal Panama") == true, "isPalindrome: Sentence");
        verify(StringUtils.isPalindrome("hello") == false, "isPalindrome: Non-palindrome");
        verify(StringUtils.isPalindrome("") == true, "isPalindrome: Empty string");
        verify(StringUtils.isPalindrome(null) == false, "isPalindrome: Null input");

        // --- reverse Tests ---
        verify("dlrow olleh".equals(StringUtils.reverse("hello world")), "reverse: standard string");
        verify("".equals(StringUtils.reverse("")), "reverse: empty string");
        verify(StringUtils.reverse(null) == null, "reverse: null input");

        // --- isAnagram Tests ---
        verify(StringUtils.isAnagram("listen", "silent") == true, "isAnagram: standard match");
        verify(StringUtils.isAnagram("Triangle", "Integral") == true, "isAnagram: case insensitive match");
        verify(StringUtils.isAnagram("apple", "pale") == false, "isAnagram: length mismatch");
        verify(StringUtils.isAnagram("", "") == true, "isAnagram: empty strings");
        verify(StringUtils.isAnagram(null, "abc") == false, "isAnagram: first string null");
        verify(StringUtils.isAnagram("abc", null) == false, "isAnagram: second string null");
        verify(StringUtils.isAnagram(null, null) == false, "isAnagram: both strings null");

        // --- Execution Summary ---
        System.out.println("\n=== TEST SUMMARY ===");
        System.out.println("Tests Found: " + (passed + failed));
        System.out.println("Tests Succeeded: " + passed);
        System.out.println("Tests Failed: " + failed);
    }

    private static void verify(boolean condition, String testName) {
        if (condition) {
            passed++;
        } else {
            failed++;
            System.err.println("FAIL: " + testName);
        }
    }
}

// 3. Execution Block for Jupyter Notebook
StringUtilsTestRunner.runTests();

=== RUNNING STRINGUTILS TEST SUITE ===

=== TEST SUMMARY ===
Tests Found: 16
Tests Succeeded: 16
Tests Failed: 0
